# Transformations - Wide vs Narrow

In [ ]:
customer_data = [
"customer_id,name,city,state,country,registration_date,is_active",
"0,Customer_0,Pune,Maharashtra,India,6/29/2023,True",
"1,Customer_1,Bangalore,Tamil Nadu,India,12/7/2023,False",
"2,Customer_2,Hyderabad,Gujarat,India,10/27/2023,True",
"3,Customer_3,Bangalore,Karnataka,India,10/17/2023,False",
"4,Customer_4,Ahmedabad,Karnataka,India,3/14/2023,True",
"5,Customer_5,Hyderabad,Karnataka,India,7/28/2023,False" ]


In [ ]:
spark = SparkSession.builder\
.appName("Transformation-Wide and Narrow")\
.getOrCreate()

In [ ]:
data_rdd = spark.sparkContext.parallelize(customer_data)

In [ ]:
header = data_rdd.first()

In [ ]:
data_rdd = data_rdd.filter(lambda row:row!=header)

In [ ]:
data_rdd.collect()

In [ ]:
cities_rdd = data_rdd.map(lambda row:(row.split(',')[2],1))

In [ ]:
cities_rdd.collect()

In [ ]:
customers_per_city = cities_rdd.reduceByKey(lambda x,y:x+y)

In [ ]:
customers_per_city.take(5)

In [ ]:
spark.stop()

# Spark Job,Stages and Tasks 

In [ ]:
spark = SparkSession.builder\
.appName("Spark-Job Stages and Tasks ")\
.getOrCreate()

In [ ]:
customer_data_without_header = ['0,Customer_0,Pune,Maharashtra,India,6/29/2023,True',
 '1,Customer_1,Bangalore,Tamil Nadu,India,12/7/2023,False',
 '2,Customer_2,Hyderabad,Gujarat,India,10/27/2023,True',
 '3,Customer_3,Bangalore,Karnataka,India,10/17/2023,False',
 '4,Customer_4,Ahmedabad,Karnataka,India,3/14/2023,True',
 '5,Customer_5,Hyderabad,Karnataka,India,7/28/2023,False']

In [ ]:
data_without_header_rdd = spark.sparkContext.parallelize(customer_data_without_header)

In [ ]:
data_without_header_rdd.getNumPartitions()

In [ ]:
data_without_header_rdd.map(lambda row:(row.split(',')[2],1)).reduceByKey(lambda x,y:x+y).count()

# ReduceByKey and GroupByKey

In [1]:
spark = SparkSession.builder\
.appName("ReduceByKey and GroupByKey")\
.getOrCreate()

26/04/19 11:57:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
!hadoop fs -ls /tmp/ 

Found 4 items
drwxr-xr-x   - root          hadoop          0 2026-04-18 18:46 /tmp/active_cities
-rw-r--r--   2 mercy16samoei hadoop    1060750 2026-04-18 15:19 /tmp/customers.csv
drwxrwxrwt   - hdfs          hadoop          0 2026-03-17 10:41 /tmp/hadoop-yarn
drwx-wx-wx   - hive          hadoop          0 2026-03-17 10:41 /tmp/hive


In [3]:
hdfs_path = '/tmp/customers.csv'
rdd = spark.sparkContext.textFile(hdfs_path)

In [4]:
header = rdd.first()

In [6]:
rdd_no_header = rdd.filter(lambda row:row!=header).map(lambda row:row.split(','))

In [7]:
rdd_no_header.first()

['0', 'Customer_0', 'Pune', 'Maharashtra', 'India', '2023-06-29', 'False']

In [8]:
reduced_rdd = rdd_no_header.map(lambda row:(row[2],1)).reduceByKey(lambda x,y:x+y)

In [9]:
reduced_rdd.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

In [10]:
grouped_rdd = rdd_no_header.map(lambda row:(row[2],1)).groupByKey()

In [11]:
grouped_rdd.collect()

[('Pune', <pyspark.resultiterable.ResultIterable at 0x7f13940a9850>),
 ('Hyderabad', <pyspark.resultiterable.ResultIterable at 0x7f13940a98d0>),
 ('Mumbai', <pyspark.resultiterable.ResultIterable at 0x7f13940a9990>),
 ('Delhi', <pyspark.resultiterable.ResultIterable at 0x7f13941419d0>),
 ('Bangalore', <pyspark.resultiterable.ResultIterable at 0x7f13940a9a10>),
 ('Ahmedabad', <pyspark.resultiterable.ResultIterable at 0x7f13baf21a50>),
 ('Chennai', <pyspark.resultiterable.ResultIterable at 0x7f13940a9b90>),
 ('Kolkata', <pyspark.resultiterable.ResultIterable at 0x7f13baf21b10>)]

In [12]:
grouped_by_result = grouped_rdd.map(lambda row:(row [0], len(row[1])))

In [13]:
grouped_by_result.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

# Repartition and Coalesce

In [1]:
spark = SparkSession.builder\
.appName("Repartition and Coalesce")\
.getOrCreate()

26/04/19 13:03:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
rdd = spark.sparkContext.textFile('/tmp/customers.csv')

In [3]:
rdd.take(5)

['customer_id,name,city,state,country,registration_date,is_active',
 '0,Customer_0,Pune,Maharashtra,India,2023-06-29,False',
 '1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True',
 '2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True',
 '3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False']

In [4]:
rdd.getNumPartitions()

2

In [6]:
repartition_rdd = rdd.repartition(4)

In [7]:
repartition_rdd.getNumPartitions()

4

In [8]:
repartition_rdd_less = rdd.repartition(1)

In [9]:
repartition_rdd_less.getNumPartitions()

1

In [11]:
coalesce_rdd = rdd.coalesce(1)

In [12]:
coalesce_rdd.getNumPartitions()

1

In [14]:
rdd.coalesce(10).getNumPartitions()

2

## Spark Higher Level APIs

In [15]:
spark = SparkSession.builder\
.appName("Spark Higher Level APIs")\
.getOrCreate()

26/04/19 13:30:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [17]:
df = spark.read.format('csv')\
.option('header','true')\
.option('inferSchema','true')\
.load('/tmp/customers.csv')

In [18]:
print(df.show(5))

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows

None


In [19]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [20]:
df.createOrReplaceTempView('customers')

In [21]:
result = spark.sql('select city,count(*) from customers group by city')

In [23]:
result.show()

+---------+--------+
|     city|count(1)|
+---------+--------+
|    Delhi|    2200|
|  Kolkata|    2223|
|Hyderabad|    2242|
|Bangalore|    2211|
|Ahmedabad|    2198|
|  Chennai|    2194|
|   Mumbai|    2142|
|     Pune|    2243|
+---------+--------+

